# Strata を Google Colab で使用する手順

このノートブックは、Google Colab 上で Strata 推論エンジンを動かし、Qwen3.8-Flash-Next などの大規模モデルを利用するための手順です。

**想定環境:**
- Google Colab (L4 または A100 GPU 推奨)
- 約 70 GB のディスク空き容量（モデルダウンロード用）

## 1. Python 3.12 の準備

Colab のデフォルト Python 3.13 では Strata のセットアップに失敗することがあるため、Python 3.12 を使用します。

In [ ]:
# Python 3.12 を優先的に使えるようにする
!mkdir -p /content/pybin
!ln -sf /usr/bin/python3.12 /content/pybin/python3
import os
os.environ['PATH'] = '/content/pybin:/usr/bin:/usr/local/cuda/bin:' + os.environ['PATH']
!python3 --version

## 2. Strata のクローンとセットアップ

GitHub から Strata をクローンし、モデル（Qwen3.8-Flash-Next IQ2_XS）をダウンードします。

- `--family qwen`: Qwen ファミリーを指定
- `--model IQ2_XS`: 軽量版を選択（L4 24GB VRAM 向け）
- `--context 131072`: コンテキスト長
- `--no-start`: セットアップのみ実行し、まだ起動しない

In [ ]:
# Strata のクローン
!cd /content && git clone --depth 1 https://github.com/Niko1221/Strata.git
%cd /content/Strata

In [ ]:
# セットアップ実行（モデルダウンロード含む、約20〜30分かかります）
!./setup.sh --yes --family qwen --model IQ2_XS --context 131072 --no-start

## 3. API サーバーの起動

Colab は 8080 番ポートを自身で使用していることが多いため、**8000 番ポート**で起動します。`nohup` を使うことで、セル実行後もバックグラウンドでサーバーが動作し続けます。

In [ ]:
# 設定ファイル名を自動取得
import subprocess
result = subprocess.run(['ls', 'strata-*.json'], capture_output=True, text=True, cwd='/content/Strata')
print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
# サーバー起動（8000番ポート）
!cd /content/Strata && CFG=$(ls strata-*.json | head -1) && nohup .venv/bin/python serve/server.py --engine strata --config "$CFG" --port 8000 --host 127.0.0.1 > /content/server.log 2>&1 &

## 4. 動作確認

サーバーが起動するまで少し待ってから、ヘルスチェックを行います。

In [ ]:
import time, requests

# 起動待ち（初回は10〜20秒程度）
time.sleep(15)

try:
    r = requests.get('http://127.0.0.1:8000/health')
    print('Status:', r.status_code)
    print('Response:', r.text)
except Exception as e:
    print('Error:', e)
    print('サーバーがまだ起動していない可能性があります。数秒待って再実行してください。')

## 5. OpenAI 互換 API でチャットを試す（オプション）

Strata は OpenAI 互換の `/v1/chat/completions` エンドポイントを提供します。以下は Python から利用する例です。

In [ ]:
import requests, json

url = 'http://127.0.0.1:8000/v1/chat/completions'
headers = {'Content-Type': 'application/json'}
payload = {
    'model': 'qwen3.8-flash-next-iq2_xs',
    'messages': [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '日本の首都はどこですか？'}
    ],
    'temperature': 0.7,
    'max_tokens': 512
}

response = requests.post(url, headers=headers, data=json.dumps(payload))
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

## 6. サーバーログの確認

もし動作がおかしい場合は、ログを確認してください。

In [ ]:
!tail -n 50 /content/server.log

## 7. （参考）Colab CLI を使った運用方法

継続的に利用したい場合は、Windows + WSL2 環境で [NeviumX/strata-colab-cli](https://github.com/NeviumX/strata-colab-cli) を使うと、ローカルからワンコマンドで起動・停止できます。

```bash
# WSL2 内で
git clone https://github.com/NeviumX/strata-colab-cli.git
cd strata-colab-cli
bash ./setup-wsl.sh
./up.sh   # 起動
./down.sh # 停止（課金を止める）
```

**注意:** 停止しないと Colab クレジットが消費され続けます。

## 注意点・トラブルシューティング

1. **Python バージョン:** 必ず Python 3.12 を使用してください。3.13 では venv 作成に失敗することがあります。
2. **ポート競合:** Colab は 8080 を占有していことがあるため、8000 など別ポートを使ってください。
3. **ディスク容量:** モデルは約 68 GB 必要です。Colab のセッションストレージ容量に注意してください。
4. **GPU メモリ:** L4 (24GB) では `IQ2_XS` など軽量量子化を選んでください。A100 (40GB) なら `IQ3_S` も動作します。
5. **課金:** Colab Pro / Pro+ など GPU 利用可能なプランが必要です。使い終わったらランタイムを停止してください。